In [1]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

credit = fetch_ucirepo(id=350)

X = credit.data.features.copy()
y = credit.data.targets.copy()

column_mapping = {
    "X1": "LIMIT_BAL",
    "X2": "SEX",
    "X3": "EDUCATION",
    "X4": "MARRIAGE",
    "X5": "AGE",
    "X6": "PAY_0",
    "X7": "PAY_2",
    "X8": "PAY_3",
    "X9": "PAY_4",
    "X10": "PAY_5",
    "X11": "PAY_6",
    "X12": "BILL_AMT1",
    "X13": "BILL_AMT2",
    "X14": "BILL_AMT3",
    "X15": "BILL_AMT4",
    "X16": "BILL_AMT5",
    "X17": "BILL_AMT6",
    "X18": "PAY_AMT1",
    "X19": "PAY_AMT2",
    "X20": "PAY_AMT3",
    "X21": "PAY_AMT4",
    "X22": "PAY_AMT5",
    "X23": "PAY_AMT6"
}

df = X.rename(columns=column_mapping).copy()
df["default"] = y["Y"].astype(int)

print(df.shape)
display(df.head())

(30000, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [2]:
df_clean = df.copy()

# EDUCATION:
# 1 = Graduate school
# 2 = University
# 3 = High school
# 4 = Other / undocumented
df_clean["EDUCATION"] = df_clean["EDUCATION"].replace({
    0: 4,
    5: 4,
    6: 4
})

# MARRIAGE:
# 1 = Married
# 2 = Single
# 3 = Other / undocumented
df_clean["MARRIAGE"] = df_clean["MARRIAGE"].replace({
    0: 3
})

print("Cleaning completed.")

Cleaning completed.


In [3]:
print("EDUCATION after cleaning:")
print(df_clean["EDUCATION"].value_counts().sort_index())

print("\nMARRIAGE after cleaning:")
print(df_clean["MARRIAGE"].value_counts().sort_index())

print("\nDataset shape after cleaning:")
print(df_clean.shape)

EDUCATION after cleaning:
EDUCATION
1    10585
2    14030
3     4917
4      468
Name: count, dtype: int64

MARRIAGE after cleaning:
MARRIAGE
1    13659
2    15964
3      377
Name: count, dtype: int64

Dataset shape after cleaning:
(30000, 24)


In [4]:
print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))

print(
    "Target unchanged:",
    df["default"].equals(df_clean["default"])
)

print(
    "Missing values after cleaning:",
    df_clean.isnull().sum().sum()
)

Original rows: 30000
Cleaned rows: 30000
Target unchanged: True
Missing values after cleaning: 0


## Data Cleaning Decisions

Based on the initial data inspection:

1. No missing-value imputation was required.
2. Undocumented EDUCATION codes (0, 5, and 6) were grouped into the "Other" category.
3. Undocumented MARRIAGE code 0 was grouped into the "Other" category.
4. Repayment-status variables were retained in their original form, including negative status codes.
5. Numerical observations were not removed solely on the basis of large or negative values.
6. No observations were removed during this cleaning stage.

In [5]:
df_clean.to_csv(
    "data/credit_card_default_clean.csv",
    index=True,
    index_label="row_id"
)

print("Cleaned dataset saved.")

Cleaned dataset saved.


In [6]:
X_clean = df_clean.drop(columns=["default"])
y_clean = df_clean["default"]

print("X shape:", X_clean.shape)
print("y shape:", y_clean.shape)

X shape: (30000, 23)
y shape: (30000,)


In [7]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    df_clean.index,
    test_size=0.20,
    random_state=42,
    stratify=df_clean["default"]
)

print("Training samples:", len(train_idx))
print("Test samples:", len(test_idx))

Training samples: 24000
Test samples: 6000


In [8]:
train_df = df_clean.loc[train_idx].copy()
test_df = df_clean.loc[test_idx].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (24000, 24)
Test shape: (6000, 24)


In [9]:
print("Overall default rate:")
print(df_clean["default"].value_counts(normalize=True).sort_index())

print("\nTraining default rate:")
print(train_df["default"].value_counts(normalize=True).sort_index())

print("\nTest default rate:")
print(test_df["default"].value_counts(normalize=True).sort_index())

Overall default rate:
default
0    0.7788
1    0.2212
Name: proportion, dtype: float64

Training default rate:
default
0    0.778792
1    0.221208
Name: proportion, dtype: float64

Test default rate:
default
0    0.778833
1    0.221167
Name: proportion, dtype: float64


In [10]:
print("Training class counts:")
print(train_df["default"].value_counts().sort_index())

print("\nTest class counts:")
print(test_df["default"].value_counts().sort_index())

Training class counts:
default
0    18691
1     5309
Name: count, dtype: int64

Test class counts:
default
0    4673
1    1327
Name: count, dtype: int64


In [11]:
train_df.to_csv(
    "data/splits/train.csv",
    index=True,
    index_label="row_id"
)

test_df.to_csv(
    "data/splits/test.csv",
    index=True,
    index_label="row_id"
)

print("Train and test datasets saved successfully.")

Train and test datasets saved successfully.


In [12]:
pd.Series(train_idx, name="row_id").to_csv(
    "data/splits/train_indices.csv",
    index=False
)

pd.Series(test_idx, name="row_id").to_csv(
    "data/splits/test_indices.csv",
    index=False
)

print("Split indices saved.")

Split indices saved.


## Train-Test Split

The cleaned dataset was divided into a training set (80%) and a test set (20%) using a stratified split.

- Training observations: 24,000
- Test observations: 6,000
- Stratification was applied to preserve the default/non-default class distribution.
- `random_state=42` was fixed for reproducibility.
- The same train-test split will be used for Logistic Regression, Decision Tree, Random Forest, and XGBoost to ensure a fair model comparison.
- Model-specific transformations such as feature scaling will be fitted using the training data only to avoid data leakage.